In [1]:
import json
import os
import time
from openai import OpenAI


In [3]:
#load dotenv
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
INPUT_FILE = '../content/questions_dataset.jsonl'
OUTPUT_FILE = '../content/questions_dataset_cleaned.jsonl'
MODEL = "openai/gpt-4o-mini" # or "google/gemini-2.0-flash-001"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    default_headers={
        "HTTP-Referer": "http://localhost:3000", # Optional: your site URL
        "X-Title": "LaTeX Fixer Script",         # Optional: your app name
    }
)

SYSTEM_PROMPT = (
    "You are a LaTeX formatting expert. Your task is to rewrite the provided text "
    "EXACTLY as it is, without changing any words, meanings, or punctuation. "
    "The ONLY changes you are permitted to make are:\n"
    "1. Convert mathematical symbols (like η, α, θ, etc.) into their proper LaTeX commands (e.g., \\eta, \\alpha, \\theta).\n"
    "2. Ensure all mathematical expressions and symbols are correctly wrapped in $ for inline math or $$ for block math.\n"
    "3. Fix any broken LaTeX delimiters (e.g., matching opening/closing $$).\n"
    "Return only the corrected text."
)


In [5]:
def fix_latex(text):
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": text}
            ],
            temperature=0, # Keep it deterministic to prevent hallucinations
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Error processing text: {e}")
        return text # Return original if it fails

def main():
    if not os.path.exists(INPUT_FILE):
        print(f"Error: {INPUT_FILE} not found.")
        return

    with open(INPUT_FILE, 'r', encoding='utf-8') as infile, \
         open(OUTPUT_FILE, 'w', encoding='utf-8') as outfile:
        
        lines = infile.readlines()
        total = len(lines)
        
        for i, line in enumerate(lines):
            try:
                data = json.loads(line)
                original_question = data.get("question", "")
                
                print(f"[{i+1}/{total}] Processing question...")
                
                # Update the question field
                data["question"] = fix_latex(original_question)
                
                # Write updated JSON object to the new file
                outfile.write(json.dumps(data, ensure_ascii=False) + "\n")
                
            except json.JSONDecodeError:
                print(f"Skipping malformed line: {i+1}")
            
            # Tiny sleep to avoid aggressive rate limiting on free/tier-1 keys
            time.sleep(0.1)

    print(f"\nProcessing complete! Saved to {OUTPUT_FILE}")

main()

[1/66] Processing question...
[2/66] Processing question...


KeyboardInterrupt: 